In [3]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()


model= ChatOllama(
    model="phi3:latest",
    temperature=0.5,
    task="text generation"
)

parser = StrOutputParser()

class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))



Thank you so much for your kind words and the time taken to provide such thoughtful feedback. I am genuinely grateful that my work met, or even exceeded, your expectations. Please do not hesitate to share any other thoughts with me in future endeavors as well! Your support means a lot to our team's motivation and drive for excellence.


In [4]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     


In [ ]:
from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

load_dotenv()

model1 = ChatOllama(
    model="phi3:latest",
    temperature=0.5,
    task="text generation"
)

model2 = ChatAnthropic(model_name='claude-3-7-sonnet-20250219')

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})


In [6]:
chain.get_graph().print_ascii()

            +---------------------------+            
            | Parallel<notes,quiz>Input |            
            +---------------------------+            
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                  +---------------+  
  | ChatOllama |                  | ChatAnthropic |  
  +------------+                  +---------------+  
          *                               *          
          *                               *          
          *                               *          
+-----------------+         

In [8]:
print(result)

NameError: name 'result' is not defined

In [10]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field

load_dotenv()

model = ChatOllama(
    model="phi3:latest",
    temperature=0.5,
    task="text generation"
)
# schema
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}


structured_model = model.with_structured_output(json_schema)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result)

{'key_themes': ['Performance', 'Camera Quality', 'Battery Life'], 'summary': 'The Samsung Galaxy S24 Ultra is a top-notch device with an insanely powerful processor and stunning camera quality. The battery life, bolstered by fast charging capabilities, ensures it keeps up throughout the day.', 'sentiment': 'pos', 'pros': ['Snapdragon 8 Gen 3 processor for lightning-fast performance in gaming and multitasking.'], 'cons': ["Device' end heaviness makes it uncomfortable to use one hand.", 'Bloatware from Samsung is unnecessary when Google apps suffice.', 'Price of $1,300 may be a hard sell for some consumers.']}


In [16]:
print(result)

key_themes=['Performance', 'Camera Quality'] summary='The Samsung Galaxy S24 Ultra impresses with its top-notch performance, thanks to the new Snapdragon 8 Gen 3 processor. Gaming and multitasking are seamless experiences here, as is photo editing on this behemoth of a phone.' sentiment='pos' pros=['Insanely powerful processor (great for gaming and productivity)', 'Stunning 2term camera with incredible zoom capabilities'] cons=['Weight and size make it hard to one-handedly use the phone.', "Samsung's One UI still comes with bloatware which is unnecessary when Google provides similar apps. The $1,300 price tag also seems steep."] name=None


In [13]:
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Literal
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

load_dotenv()

# Define Pydantic model
class Sentiment(BaseModel):
    sentiment: Literal["POSITIVE", "NEGATIVE", "NEUTRAL"]
    confidence: float

# Initialize model
model = ChatOllama(
    model="phi3:latest",
    temperature=0.5,
    task="text generation"
)

# Create structured output
structured_model = model.with_structured_output(Sentiment)

# Analyze sentiment
review = "I absolutely love this product! It works perfectly."

result = structured_model.invoke(f"Analyze the sentiment of this review: {review}")

print(f"Review: {review}")
print(f"Sentiment: {result.sentiment}")
print(f"Confidence: {result.confidence}")

Review: I absolutely love this product! It works perfectly.
Sentiment: POSITIVE
Confidence: 0.95


In [14]:
import os
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# Initialize Groq model (you need to set GROQ_API_KEY in .env file)
model1 = ChatOllama(
    model="phi3:latest",
    temperature=0.5,
    task="text generation"
)


# Schema definition (same as yours)
class Review(BaseModel):
    key_themes: list[str] = Field(description="Write down all the key themes discussed in the review in a list")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(description="Return sentiment of the review either negative, positive or neutral")
    pros: Optional[list[str]] = Field(default=None, description="Write down all the pros inside a list")
    cons: Optional[list[str]] = Field(default=None, description="Write down all the cons inside a list")
    name: Optional[str] = Field(default=None, description="Write the name of the reviewer")

# Create structured output model
structured_model = model.with_structured_output(Review)

# Sample review text
review_text = """I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it's an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I'm gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung's One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh"""

# Invoke the model
result = structured_model.invoke(review_text)

# Print results
print("\n" + "="*50)
print("STRUCTURED REVIEW ANALYSIS")
print("="*50)
print(f"Key Themes: {result.key_themes}")
print(f"\nSummary: {result.summary}")
print(f"\nSentiment: {result.sentiment}")
print(f"\nPros: {result.pros}")
print(f"\nCons: {result.cons}")
print(f"\nReviewer: {result.name}")


STRUCTURED REVIEW ANALYSIS
Key Themes: ['Performance', 'Camera Quality']

Summary: The Samsung Galaxy S24 Ultra impresses with its top-notch performance, thanks to the new Snapdragon 8 Gen 3 processor. Gaming and multitasking are seamless experiences here, as is photo editing on this behemoth of a phone.

Sentiment: pos

Pros: ['Insanely powerful processor (great for gaming and productivity)', 'Stunning 2term camera with incredible zoom capabilities']

Cons: ['Weight and size make it hard to one-handedly use the phone.', "Samsung's One UI still comes with bloatware which is unnecessary when Google provides similar apps. The $1,300 price tag also seems steep."]

Reviewer: None
